# Does RFM Really Predict the Future?
### 거래처 고객의 재주문을 세 가지 방법으로 비교하기

**저자:** 이창민 (A) · 서유나 (B) · 이정민 (C)

> **이 노트북은 제출용 글의 뼈대다.** 각 장의 "담당"이 자기 노트북에서 완성한 글과 코드를 해당 장으로 옮긴다. 옮긴 뒤 이 노트북은 **위에서부터 한 번에 실행**되어야 한다 (`Kernel → Restart & Run All`). 이 안내 문구와 각 장의 `TODO`는 제출 전에 지운다.

In [1]:
# 공통 준비: 모든 장이 이 셀의 결과를 쓴다
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rfm_utils as ru

clean, report = ru.clean_transactions(ru.load_raw())
inv = ru.invoice_table(clean)
datasets = {name: ru.make_dataset(inv, snap) for name, snap in ru.SNAPSHOTS.items()}
for name, snap in ru.SNAPSHOTS.items():
    ru.assert_no_leakage(inv, datasets[name], snap)

## 1. 도입: RFM은 정말 미래를 예측하는가? (담당: C)
- 질문: 최근성·빈도·금액 세 숫자로 "앞으로 90일 안에 다시 주문할 거래처"를 얼마나 맞힐 수 있는가
- 왜 중요한가: 한정된 영업·재주문 유도 예산을 누구에게 먼저 쓸 것인가
- 이 글이 하는 일(세 방법 비교)과 결론 한 문장 미리 보기
- 독자가 얻어 갈 것: RFM, 시간 분할(누수 방지), 가설 검정, 평가 지표

*TODO: 이 장의 글 (약 300단어)*

## 2. 데이터: 거래처 고객의 구매 기록 (담당: A)
- Online Retail II 소개, 이 데이터가 소비자가 아니라 **도매 성격의 거래처**라는 근거(주문 중앙값, 상품 한 줄 수량)
- 정제 규칙과 이유, 정제 보고서 표
- EDA: 월별 매출, R·F·M 분포, 상위 10% 고객의 매출 비중
- 기준일 S1·S2·S3와 시간 분할, 누수 방지 설명

*TODO: A 노트북의 1~2절과 EDA 그림·해석을 옮긴다*

In [2]:
# TODO (A): 정제 보고서와 EDA 코드
report

,step,rows
0,raw rows,1067371
1,removed: cancellations (invoice starts with C),19494
2,removed: missing customer id,242257
3,removed: quantity <= 0 or price <= 0,71
4,"removed: non-product lines (postage, fees, man...",2915
5,removed: exact duplicate rows,26055
6,clean rows,776579


## 3. 방법 ①: 규칙 기반 RFM 세그먼트와 가설 H1·H2 (담당: B)
- 5분위 점수, 세그먼트 규칙표, 세그먼트별 재구매율·매출
- H1: 세그먼트별 재구매율 차이(카이제곱, Cramér's V), F·M이 R 이상의 정보를 주는가
- H2: 세그먼트별 라벨 구간 매출 차이(Kruskal-Wallis + Dunn)
- 보조: K-means vs 규칙 세그먼트(ARI), 세 기준일에서의 안정성

*TODO: B 노트북의 해당 절을 옮긴다*

In [3]:
# TODO (B): 세그먼트, H1, H2, K-means 코드

## 4. 방법 ②: 머신러닝과 가설 H3 (담당: C)
- 로지스틱 → 랜덤 포레스트/그래디언트 부스팅, S1 학습·S2 검증·S3 테스트
- H3: 기준선("최근에 샀는가"만)보다 나은가 (AUC·PR-AUC, 부트스트랩 신뢰구간)
- 중요한 특징과 해석

*TODO: C 노트북의 해당 절을 옮긴다*

In [4]:
# TODO (C): 모델 학습과 H3 코드

## 5. 방법 ③: BG/NBD (도전 파트, 담당: C)
- BG/NBD가 가정하는 것(구매가 무작위·포아송)과 이 데이터의 차이: 거래처 재주문은 재고를 보고 의도해서 하는 구매라 간격이 더 규칙적(CV 중앙값 0.75)
- 적합 결과와 기대 구매 횟수 점수
- 가정이 깨진 상태에서 얼마나 버티는가

*TODO: C 노트북의 BG/NBD 절을 옮긴다. 시간이 부족하면 이 장은 짧게 줄이고 4장을 먼저 끝낸다*

In [5]:
# TODO (C): BG/NBD 코드

## 6. 세 방법 비교와 예산 시뮬레이션 (담당: A)
- 같은 고객·기준일·정답으로 AUC, PR-AUC, lift@10%, capture@10%, 이득 곡선
- H4: lift@10%가 세 방법 사이에 다른가
- 예산 시뮬레이션: 상위 k%에 연락할 때의 기대 효과 (가정은 표로 명시, 인과 주장 금지)

아래 셀은 `outputs/scores_*.csv`를 읽어 비교표를 만든다 (이미 동작한다).

In [6]:
scores = ru.load_score_frames()
rows = []
for (method, snap), g in scores.groupby(["method", "snapshot"]):
    name = [k for k, v in ru.SNAPSHOTS.items() if pd.Timestamp(v) == snap][0]
    d = datasets[name]
    s = g.set_index("customer_id")["score"].reindex(d.index)
    assert s.notna().all(), f"{method} {name}: missing customers"
    rows.append({"method": method, "snapshot": name, **ru.evaluate_scores(d["repurchase"], s)})
for name, d in datasets.items():
    rows.append({"method": "recency_only", "snapshot": name, **ru.evaluate_scores(d["repurchase"], ru.baseline_recency_score(d))})
comparison = pd.DataFrame(rows).sort_values(["snapshot", "method"]).reset_index(drop=True)
comparison[comparison["snapshot"] == "S3"].round(3)   # 최종 비교는 S3(테스트) 기준

,method,snapshot,auc,pr_auc,base_rate,capture@10,precision@10,lift@10
5,logistic,S3,0.785,0.743,0.428,0.205,0.877,2.050
6,recency_only,S3,0.760,0.671,0.428,0.178,0.761,1.777
7,rule_rfm,S3,0.782,0.723,0.428,0.210,0.900,2.104


## 7. 결론과 한계 (담당: 세 명이 함께, A가 정리)
- 질문에 대한 답: RFM은 어디까지 맞히는가, 기준선과 ML, BG/NBD는 어떻게 다른가
- 거래처 관리 관점의 시사점(어느 거래처에 먼저 영업할지)
- 한계: 소비자 일반으로 확장 불가(도매 성격), 완전 중복 행 제거 가정, 90일 단일 창, 관측 데이터라 인과 해석 금지
- 더 해 볼 것

*TODO*